# ANALYSE-07 — Critical percolation and Ising on the tree (native companion)

**ANALYSE** sub-series (#17545 grading) — Fold 3 of the **Origami** digestion (#19898): two 2026 results read from **our** `percolation_lean` lake — the Harris–Kleitman kernel and the connection event.

This notebook is a **native** companion: Lean 4 kernel under WSL, every `#check` queries the actually-compiled lake, every `#eval` really computes. Numerical computation is done **natively in Lean** — no Python fallback; the file-name suffix follows the series naming convention. This is the English twin of the French notebook `ANALYSE-07-Percolation-Critique.ipynb`; it renders the **same** computation in English: the code cells are byte-identical to the French side (the repository's twin-parity invariant), while the lake's English siblings (`*_en`, i18n convention #4980) carry the English docstrings.

**Prerequisites**: the main series tutorial (notebooks 1 to 6). ANALYSE-04 (PFR primitives) gives a feel for inequality arguments on a Boolean lattice; it is not required.

**Plan**: our own foundation (Harris–Kleitman, FKG inequalities) — the two 2026 families quoted with their status — our computation of percolation on the $d$-regular tree (Galton–Watson, exact) — the distinction between two thresholds that are often conflated — the lake's finite objects — three exercises.

## Navigation

| Previous notebook | French version | Next notebook |
|---|---|---|
| [ANALYSE-04 — Three PFR primitives](ANALYSE-04-PFR-Primitives-Python.ipynb) | [ANALYSE-07 — French version](ANALYSE-07-Percolation-Critique.ipynb) | [Lean-12b — Huang's Sensitivity Theorem](../Lean-12b-Lean-Sensitivity-Theorem.ipynb) |

## 0. Epistemic status — the rule of this notebook

The Origami digestion requires a **declared status** for every statement. Three statuses exist: **ours** (proved or computed in our lake), **formalised** (proved by the openai/math authors, Lean skeleton published), **not formalised** (no formal proof). The rule: never "OpenAI proved X, therefore X" — this notebook distinguishes at every line what **we prove and execute** from what **we quote**.

| Object | Status | Where to check it |
|---|---|---|
| `harris_kleitman_upper_upper`, `harris_kleitman_lower_lower` (FKG inequalities, lake `percolation_lean`) | **ours** | `#check` + `#print axioms`, section 1 |
| Family **213** — Benjamini–Schramm criticality | **formalised** (by the authors) | quoted section 2 — skeleton `lean/docs/213.md` of openai/math |
| Family **236** — factor-of-IID threshold of Ising on the $d$-regular tree | **formalised** (by the authors) | quoted section 4 — skeleton `lean/docs/236.md` |
| Critical probability $p_c = 1/(d-1)$ on the $d$-regular tree; reconstruction threshold $\tanh\beta_c = (d-1)^{-1/2}$ | **our computation** (Galton–Watson, elementary) | sections 3 and 4 — native `#eval` |

What this notebook **does not do**: re-prove the 2026 results. Their scale puts re-proving beyond this repository's reach — which is precisely why the "formalised" status exists.

**A vocabulary trap, defused up front.** Two thresholds circulate for Ising on the $d$-regular tree and they are **not the same quantity**: the **reconstruction** threshold (Kesten–Stigum, what family 236 states), $\tanh\beta_c = (d-1)^{-1/2}$; and the **spontaneous magnetisation** threshold, $\tanh\beta_c = 1/(d-1)$. Section 4 computes both and shows the gap. Conflating them is the mistake this notebook exists to prevent.

## 1. The foundation: the Harris–Kleitman inequalities, proved in our lake

The `percolation_lean` lake carries the combinatorial kernel of percolation theory: the **FKG inequalities** (Fortuin–Kasteleyn–Ginibre) in their most elementary case — increasing and decreasing families of a Boolean lattice.

The statement: if $\mathcal{A}$ is an **increasing** family (adding an element can only preserve membership) and $\mathcal{B}$ a **decreasing** one, then
$$|\mathcal{A}| \cdot |\mathcal{B}| \;\le\; 2^{|\alpha|} \cdot |\mathcal{A} \cap \mathcal{B}|.$$
This is the Harris–Kleitman inequality, and it is exactly the tool that makes percolation monotone: the event "$u$ is connected to $v$" is increasing, the event "$\mathcal{A}$ is closed" is decreasing, and their positive correlation is what gives meaning to "there exists a threshold".

Let us import the lake. Its globs cover the **sub-modules** (not the root): we therefore import explicitly the modules this notebook uses — the root aggregator `Percolation.lean` is not a compiled module of the lake:

In [1]:
import Percolation.Basic
import Percolation.Boundary
import Percolation.Components
import Percolation.Connectivity
import Percolation.Examples

open Percolation

#check Percolation.harris_kleitman_upper_upper
#check Percolation.harris_kleitman_lower_lower
#check Percolation.harris_kleitman_connected

import Percolation.Basic
import Percolation.Boundary
import Percolation.Components
import Percolation.Connectivity
import Percolation.Examples

open Percolation

#check Percolation.harris_kleitman_upper_upper
──────▶  Percolation.harris_kleitman_upper_upper.{u_1} {α : Type u_1} [Fintype α] [DecidableEq α] (𝒜 ℬ : Finset (Finset α))
  (h𝒜 : IsUpperSet ↑𝒜) (hℬ : IsUpperSet ↑ℬ) : 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card
#check Percolation.harris_kleitman_lower_lower
──────▶  Percolation.harris_kleitman_lower_lower.{u_1} {α : Type u_1} [Fintype α] [DecidableEq α] (𝒜 ℬ : Finset (Finset α))
  (h𝒜 : IsLowerSet ↑𝒜) (hℬ : IsLowerSet ↑ℬ) : 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card
#check Percolation.harris_kleitman_connected
──────▶  Percolation.harris_kleitman_connected.{u_1} {V : Type u_1} [Fintype V] [DecidableEq V] (G : SimpleGraph V)
  [Fintype (Edge G)] [DecidableEq (Edge G)] {u v x y : V} :
  (connectedEvent G u v).card * (connectedEvent G x y).card ≤
    2 ^ Fintype.card (Edge G) * (connectedEvent G u v ∩ connectedEvent G x y).card
--% env 0

Raw input:
{"cmd": "import Percolation.Basic\nimport Percolation.Boundary\nimport Percolation.Components\nimport Percolation.Connectivity\nimport Percolation.Examples\n\nopen Percolation\n\n#check Percolation.harris_kleitman_upper_upper\n#check Percolation.harris_kleitman_lower_lower\n#check Percolation.harris_kleitman_connected"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "Percolation.harris_kleitman_upper_upper.{u_1} {α : Type u_1} [Fintype α] [DecidableEq α] (𝒜 ℬ : Finset (Finset α))\n  (h𝒜 : IsUpperSet ↑𝒜) (hℬ : IsUpperSet ↑ℬ) : 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "Percolation.harris_kleitman_lower_lower.{u_1} {α : Type u_1} [Fintype α] [DecidableEq α] (𝒜 ℬ : Finset (Finset α))\n  (h𝒜 : IsLowerSet ↑𝒜) (hℬ : IsLowerSet ↑ℬ) : 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "Percolation.harris_kleitman_connected.{u_1} {V : Type u_1} [Fintype V] [DecidableEq V] (G : SimpleGraph V)\n  [Fintype (Edge G)] [DecidableEq (Edge G)] {u v x y : V} :\n  (connectedEvent G u v).card * (connectedEvent G x y).card ≤\n    2 ^ Fintype.card (Edge G) * (connectedEvent G u v ∩ connectedEvent G x y).card"}],
 "env": 0}

### Reading the result

The first two declarations are the foundation: `harris_kleitman_upper_upper` for two increasing families, `harris_kleitman_lower_lower` for two decreasing ones. The third, `harris_kleitman_connected`, is the **percolation** version of the same kernel — it bears on the connection event, and it is the one that speaks this notebook's language. Let us verify their logical footprint:

In [2]:
#print axioms Percolation.harris_kleitman_upper_upper
#print axioms Percolation.harris_kleitman_lower_lower
#print axioms Percolation.harris_kleitman_connected

#print axioms Percolation.harris_kleitman_upper_upper
──────▶  'Percolation.harris_kleitman_upper_upper' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms Percolation.harris_kleitman_lower_lower
──────▶  'Percolation.harris_kleitman_lower_lower' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms Percolation.harris_kleitman_connected
──────▶  'Percolation.harris_kleitman_connected' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 1

Raw input:
{"cmd": "#print axioms Percolation.harris_kleitman_upper_upper\n#print axioms Percolation.harris_kleitman_lower_lower\n#print axioms Percolation.harris_kleitman_connected", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'Percolation.harris_kleitman_upper_upper' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'Percolation.harris_kleitman_lower_lower' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'Percolation.harris_kleitman_connected' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 1}

### No `sorry`, minimal footprint

`[propext, Classical.choice, Quot.sound]` — the three standard Mathlib axioms, no `sorryAx`, no `native_decide`. The kernel is **ours** in the fullest sense: proved in the lake, verifiable by this command at any time.

## 2. Family 213 — Benjamini–Schramm criticality

**Status: formalised** (by the openai/math authors, 2026). Statement quoted, never executed here.

The Benjamini–Schramm conjecture predicts the **absence of percolation at criticality**: for bond percolation on an infinite, connected, locally finite, transitive graph, if there exists a probability $p$ for which the open cluster is infinite almost surely, then the same holds at the critical probability $p_c$. In other words: the critical regime is not a "soft" transition regime — it already lies in the connected phase as soon as $p_c$ is reached.

What this changes for us: the question "what happens *exactly* at $p_c$?" is not a technical detail, it is the question that occupied percolation theory for forty years. The 2026 result quoted here settles it for this class of graphs.

**What we do not do**: neither state it as ours, nor give a proof. The `lean/docs/213.md` skeleton of the `openai/math` repository is the reference; it is not present in this repository (verified), so we do not claim to have read it — we quote the statement as the Origami distillation reports it.

## 3. Our computation — percolation on the $d$-regular tree

Here is the **our own** part, and it is exact, not approximate.

On the $d$-regular tree, we explore the open component of the root vertex through a **Galton–Watson** process: every reached vertex has $d-1$ children (the parent is already counted), and each child edge is open independently with probability $p$. The mean number of open children is therefore $(d-1)p$.

The mean-offspring criterion gives the threshold: the component is finite if $(d-1)p < 1$, and infinite (with positive probability) if $(d-1)p > 1$. Hence
$$p_c = \frac{1}{d-1}.$$

The extinction probability obeys its own recursion — the probability generating function of the binomial law $\mathrm{Bin}(d-1, p)$ at the point $q$:
$$q_{n+1} = \big((1-p) + p\,q_n\big)^{d-1}, \qquad q_0 = 0.$$
The non-trivial fixed point of this recursion is the probability that **the infection dies out before spreading**. Let us write it in native Lean:

In [3]:
/-- Puissance entiere sur `Float` : `HPow Float Nat` n'existe pas dans cette
version de Lean (mesure : `failed to synthesize HPow Float ℕ`), et un echec
d'instance rendrait l'expression dependante de l'axiome `sorry` -- `#eval`
abandonnerait alors sans calculer. On l'ecrit donc a la main. -/
def fpow (x : Float) : Nat → Float
  | 0 => 1.0
  | n + 1 => x * fpow x n

/-- Probabilite d'extinction apres `n` generations, pour `b = d - 1` enfants
par sommet. Recursion de Galton-Watson de la loi Bin(b, p). -/
def extinction (p : Float) (b : Nat) : Nat → Float
  | 0 => 0.0
  | k + 1 => fpow ((1.0 - p) + p * extinction p b k) b

-- b = 2  (d = 3) : p_c = 1/2
#eval extinction 0.30 2 80    -- sous-critique -> extinction certaine
#eval extinction 0.50 2 80    -- a p_c exactement (progeniture moyenne = 1) -> tend vers 1
#eval extinction 0.60 2 80    -- sur-critique -> survie de probabilite positive, donc q* < 1

-- b = 3  (d = 4) : p_c = 1/3
#eval extinction 0.25 3 80
#eval extinction 0.40 3 80

-- b = 4  (d = 5) : p_c = 1/4
#eval extinction 0.20 4 80
#eval extinction 0.30 4 80

/-- Puissance entiere sur `Float` : `HPow Float Nat` n'existe pas dans cette
version de Lean (mesure : `failed to synthesize HPow Float ℕ`), et un echec
d'instance rendrait l'expression dependante de l'axiome `sorry` -- `#eval`
abandonnerait alors sans calculer. On l'ecrit donc a la main. -/
def fpow (x : Float) : Nat → Float
  | 0 => 1.0
  | n + 1 => x * fpow x n

/-- Probabilite d'extinction apres `n` generations, pour `b = d - 1` enfants
par sommet. Recursion de Galton-Watson de la loi Bin(b, p). -/
def extinction (p : Float) (b : Nat) : Nat → Float
  | 0 => 0.0
  | k + 1 => fpow ((1.0 - p) + p * extinction p b k) b

-- b = 2  (d = 3) : p_c = 1/2
#eval extinction 0.30 2 80    -- sous-critique -> extinction certaine
─────▶  1.000000
#eval extinction 0.50 2 80    -- a p_c exactement (progeniture moyenne = 1) -> tend vers 1
─────▶  0.954145
#eval extinction 0.60 2 80    -- sur-critique -> survie de probabilite positive, donc q* < 1
─────▶  0.444444

-- b = 3  (d = 4) : p_c = 1/3
#eval extinction 0.25 3 80
─────▶  1.000000
#eval extinction 0.40 3 80
─────▶  0.557189

-- b = 4  (d = 5) : p_c = 1/4
#eval extinction 0.20 4 80
─────▶  1.000000
#eval extinction 0.30 4 80
─────▶  0.598335
--% env 2

Raw input:
{"cmd": "/-- Puissance entiere sur `Float` : `HPow Float Nat` n'existe pas dans cette\nversion de Lean (mesure : `failed to synthesize HPow Float \u2115`), et un echec\nd'instance rendrait l'expression dependante de l'axiome `sorry` -- `#eval`\nabandonnerait alors sans calculer. On l'ecrit donc a la main. -/\ndef fpow (x : Float) : Nat \u2192 Float\n  | 0 => 1.0\n  | n + 1 => x * fpow x n\n\n/-- Probabilite d'extinction apres `n` generations, pour `b = d - 1` enfants\npar sommet. Recursion de Galton-Watson de la loi Bin(b, p). -/\ndef extinction (p : Float) (b : Nat) : Nat \u2192 Float\n  | 0 => 0.0\n  | k + 1 => fpow ((1.0 - p) + p * extinction p b k) b\n\n-- b = 2  (d = 3) : p_c = 1/2\n#eval extinction 0.30 2 80    -- sous-critique -> extinction certaine\n#eval extinction 0.50 2 80    -- a p_c exactement (progeniture moyenne = 1) -> tend vers 1\n#eval extinction 0.60 2 80    -- sur-critique -> survie de probabilite positive, donc q* < 1\n\n-- b = 3  (d = 4) : p_c = 1/3\n#eval extinction 0.25 3 80\n#eval extinction 0.40 3 80\n\n-- b = 4  (d = 5) : p_c = 1/4\n#eval extinction 0.20 4 80\n#eval extinction 0.30 4 80", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 5},
   "data": "1.000000"},
  {"severity": "info",
   "pos": {"line": 17, "column": 0},
   "endPos": {"line": 17, "column": 5},
   "data": "0.954145"},
  {"severity": "info",
   "pos": {"line": 18, "column": 0},
   "endPos": {"line": 18, "column": 5},
   "data": "0.444444"},
  {"severity": "info",
   "pos": {"line": 21, "column": 0},
   "endPos": {"line": 21, "column": 5},
   "data": "1.000000"},
  {"severity": "info",
   "pos": {"line": 22, "column": 0},
   "endPos": {"line": 22, "column": 5},
   "data": "0.557189"},
  {"severity": "info",
   "pos": {"line": 25, "column": 0},
   "endPos": {"line": 25, "column": 5},
   "data": "1.000000"},
  {"severity": "info",
   "pos": {"line": 26, "column": 0},
   "endPos": {"line": 26, "column": 5},
   "data": "0.598335"}],
 "env": 2}

### Reading the result

Three readings, from the simplest to the most interesting.

**Sub-critical, extinction is certain.** Below $p_c$, the first three values are $1.000000$: the open component is finite almost surely. This is the regime where the cluster does not "percolate".

**At $p_c$ exactly, the value is subtle.** For $b=2$ and $p=0.5$ the mean offspring equals exactly $1$: this is the critical case, and the computation returns $0.954145$ after 80 iterations — close to $1$ without reaching it, and **converging slowly**. That slowness is not an artefact: at criticality the decay of the survival probability is polynomial, not exponential. This is precisely the regime family 213 speaks about.

**Super-critical, the fixed point is exactly $4/9$.** For $b=2$, $p=0.6$, the computation returns $0.444444$. Let us solve by hand to check: $q = (0.4 + 0.6q)^2$ gives $0.36q^2 - 0.52q + 0.16 = 0$, i.e. $9q^2 - 13q + 4 = 0$, whose roots are $q = 1$ and $q = 4/9$. The non-trivial fixed point is therefore **exactly $4/9$** — and that is the value the compiler produces, on this notebook's own definitions. A value coming out of the computation and checkable by hand: that is what one asks of a pedagogical `#eval`.

## 4. Family 236 — the Ising threshold on the tree, and the two quantities that get conflated

**Status: formalised** (by the openai/math authors, 2026). Statement quoted, never executed here.

Family 236 gives the **exact factor-of-IID threshold** for the free Ising model on the $d$-regular tree: the property holds **exactly** when
$$\tanh\beta \;\le\; (d-1)^{-1/2},$$
equality included, for $d \ge 3$ and $\beta \ge 0$ — the question, raised by Lyons, in the ferromagnetic case.

**The trap, and it is a classical one.** Another threshold circulates for the same object: $\tanh\beta_c = 1/(d-1)$. Both are real, both are thresholds of Ising on the $d$-regular tree, and **they do not measure the same thing**:

| Threshold | What it characterises | Origin |
|---|---|---|
| $\tanh\beta_c = (d-1)^{-1/2}$ | **reconstruction**: is the state of the root still correlated with that of the distant boundary? | Kesten–Stigum; this is the object of family 236 |
| $\tanh\beta_c = 1/(d-1)$ | **spontaneous magnetisation**: does the ordered phase appear? | Bethe / mean-field criterion on the tree |

Concretely, $(d-1)^{-1/2} > 1/(d-1)$ for $d \ge 3$: the reconstruction threshold is **higher**. Between the two there exists a phase where the magnetisation vanishes yet the boundary information is still reconstructible. That intermediate regime is what makes Lyons' question interesting — and it is why presenting the two under a single name would be a fault, not a simplification.

**What we do not do**: derive the quoted threshold. What follows is an **elementary computation of ours** — the algebra of the two thresholds and their gap on small $d$, executed by the compiler. It is a consistency check, **not** a re-proof of family 236.

In [4]:
-- Deux pieges de coercion, tous deux MESURES au preflight -- et tous deux du
-- meme genre : l'echec d'instance ne dit pas « erreur de type », il rend la
-- definition dependante de l'axiome `sorry`, et les `#eval` en aval AVORTENT
-- au lieu de calculer. Un faux « echec de calcul » qui est une erreur de type.
--
--   (1) `HPow Float Nat` n'existe pas : d'ou `fpow` ecrit a la main ci-dessus.
--   (2) `(d - 1 : Float)` sur `d : Nat` ne se coerced PAS (Lean cherche
--       `HSub Nat Nat Float`) ; `((d - 1 : Nat) : Float)` passe l'analyse mais
--       laisse ensuite un `Nat` la ou un `Float` est attendu.
--
-- La forme qui marche est explicite : `Float.ofNat (d - 1)` -- soustraction
-- dans Nat, puis conversion nommee. Aucune inference implicite en jeu.

/-- Seuil de reconstruction (Kesten-Stigum) : `tanh beta = (d-1)^(-1/2)`.
Egal a la racine positive de `(d-1) * t^2 = 1`. -/
def seuilReconstruction (d : Nat) : Float :=
  1.0 / Float.sqrt (Float.ofNat (d - 1))

/-- Seuil d'aimantation (critere de Bethe) : `tanh beta = 1/(d-1)`.
A ne PAS confondre avec le precedent : c'est l'objet de la colonne de droite
du tableau de la section 4. -/
def seuilAimantation (d : Nat) : Float :=
  1.0 / Float.ofNat (d - 1)

/-- Verifie l'identite algebrique `(d-1) * t^2 = 1` pour `t` le seuil de
reconstruction : c'est la forme sous laquelle Kesten-Stigum l'ecrit. -/
def verifieKS (d : Nat) : Float :=
  Float.ofNat (d - 1) * seuilReconstruction d * seuilReconstruction d

-- Les deux seuils, cote a cote, sur petits d (d >= 3 -> b = d-1 >= 2)
#eval (seuilReconstruction 3, seuilAimantation 3)   -- (0.707107, 0.500000)
#eval (seuilReconstruction 4, seuilAimantation 4)   -- (0.577350, 0.333333)
#eval (seuilReconstruction 5, seuilAimantation 5)   -- (0.500000, 0.250000)
#eval (seuilReconstruction 6, seuilAimantation 6)

-- L'identite de Kesten-Stigum, verifiee numeriquement : doit rendre 1.0
#eval verifieKS 3
#eval verifieKS 4
#eval verifieKS 5

-- Le seuil de reconstruction est STRICTEMENT au-dessus de celui d'aimantation
#eval (seuilReconstruction 3 > seuilAimantation 3)
#eval (seuilReconstruction 10 > seuilAimantation 10)

-- Deux pieges de coercion, tous deux MESURES au preflight -- et tous deux du
-- meme genre : l'echec d'instance ne dit pas « erreur de type », il rend la
-- definition dependante de l'axiome `sorry`, et les `#eval` en aval AVORTENT
-- au lieu de calculer. Un faux « echec de calcul » qui est une erreur de type.
--
--   (1) `HPow Float Nat` n'existe pas : d'ou `fpow` ecrit a la main ci-dessus.
--   (2) `(d - 1 : Float)` sur `d : Nat` ne se coerced PAS (Lean cherche
--       `HSub Nat Nat Float`) ; `((d - 1 : Nat) : Float)` passe l'analyse mais
--       laisse ensuite un `Nat` la ou un `Float` est attendu.
--
-- La forme qui marche est explicite : `Float.ofNat (d - 1)` -- soustraction
-- dans Nat, puis conversion nommee. Aucune inference implicite en jeu.

/-- Seuil de reconstruction (Kesten-Stigum) : `tanh beta = (d-1)^(-1/2)`.
Egal a la racine positive de `(d-1) * t^2 = 1`. -/
def seuilReconstruction (d : Nat) : Float :=
  1.0 / Float.sqrt (Float.ofNat (d - 1))

/-- Seuil d'aimantation (critere de Bethe) : `tanh beta = 1/(d-1)`.
A ne PAS confondre avec le precedent : c'est l'objet de la colonne de droite
du tableau de la section 4. -/
def seuilAimantation (d : Nat) : Float :=
  1.0 / Float.ofNat (d - 1)

/-- Verifie l'identite algebrique `(d-1) * t^2 = 1` pour `t` le seuil de
reconstruction : c'est la forme sous laquelle Kesten-Stigum l'ecrit. -/
def verifieKS (d : Nat) : Float :=
  Float.ofNat (d - 1) * seuilReconstruction d * seuilReconstruction d

-- Les deux seuils, cote a cote, sur petits d (d >= 3 -> b = d-1 >= 2)
#eval (seuilReconstruction 3, seuilAimantation 3)   -- (0.707107, 0.500000)
─────▶  (0.707107, 0.500000)
#eval (seuilReconstruction 4, seuilAimantation 4)   -- (0.577350, 0.333333)
─────▶  (0.577350, 0.333333)
#eval (seuilReconstruction 5, seuilAimantation 5)   -- (0.500000, 0.250000)
─────▶  (0.500000, 0.250000)
#eval (seuilReconstruction 6, seuilAimantation 6)
─────▶  (0.447214, 0.200000)

-- L'identite de Kesten-Stigum, verifiee numeriquement : doit rendre 1.0
#eval verifieKS 3
─────▶  1.000000
#eval verifieKS 4
─────▶  1.000000
#eval verifieKS 5
─────▶  1.000000

-- Le seuil de reconstruction est STRICTEMENT au-dessus de celui d'aimantation
#eval (seuilReconstruction 3 > seuilAimantation 3)
─────▶  true
#eval (seuilReconstruction 10 > seuilAimantation 10)
─────▶  true
--% env 3

Raw input:
{"cmd": "-- Deux pieges de coercion, tous deux MESURES au preflight -- et tous deux du\n-- meme genre : l'echec d'instance ne dit pas \u00ab erreur de type \u00bb, il rend la\n-- definition dependante de l'axiome `sorry`, et les `#eval` en aval AVORTENT\n-- au lieu de calculer. Un faux \u00ab echec de calcul \u00bb qui est une erreur de type.\n--\n--   (1) `HPow Float Nat` n'existe pas : d'ou `fpow` ecrit a la main ci-dessus.\n--   (2) `(d - 1 : Float)` sur `d : Nat` ne se coerced PAS (Lean cherche\n--       `HSub Nat Nat Float`) ; `((d - 1 : Nat) : Float)` passe l'analyse mais\n--       laisse ensuite un `Nat` la ou un `Float` est attendu.\n--\n-- La forme qui marche est explicite : `Float.ofNat (d - 1)` -- soustraction\n-- dans Nat, puis conversion nommee. Aucune inference implicite en jeu.\n\n/-- Seuil de reconstruction (Kesten-Stigum) : `tanh beta = (d-1)^(-1/2)`.\nEgal a la racine positive de `(d-1) * t^2 = 1`. -/\ndef seuilReconstruction (d : Nat) : Float :=\n  1.0 / Float.sqrt (Float.ofNat (d - 1))\n\n/-- Seuil d'aimantation (critere de Bethe) : `tanh beta = 1/(d-1)`.\nA ne PAS confondre avec le precedent : c'est l'objet de la colonne de droite\ndu tableau de la section 4. -/\ndef seuilAimantation (d : Nat) : Float :=\n  1.0 / Float.ofNat (d - 1)\n\n/-- Verifie l'identite algebrique `(d-1) * t^2 = 1` pour `t` le seuil de\nreconstruction : c'est la forme sous laquelle Kesten-Stigum l'ecrit. -/\ndef verifieKS (d : Nat) : Float :=\n  Float.ofNat (d - 1) * seuilReconstruction d * seuilReconstruction d\n\n-- Les deux seuils, cote a cote, sur petits d (d >= 3 -> b = d-1 >= 2)\n#eval (seuilReconstruction 3, seuilAimantati

### Reading the result

The three pairs confirm the table, and the final pair of values says why the trap deserves to be explicitly defused:

- $d = 3$: reconstruction $0.707107$, magnetisation $0.500000$;
- $d = 4$: $0.577350$ against $0.333333$;
- $d = 5$: $0.500000$ against $0.250000$.

The reconstruction threshold is **above** the magnetisation threshold for every $d \ge 3$, and the gap widens with $d$: $1/\sqrt{2}$ against $1/2$, then $1/\sqrt{3}$ against $1/3$… The two quantities are therefore not merely distinct, they are **separated by a non-empty interval** — that is where the "zero magnetisation, reconstruction possible" regime lives.

The Kesten–Stigum identity $(d-1)\,t^2 = 1$ does return $1.000000$ in all three cases: the computed threshold is exactly the positive root of that equation, which is the form in which the literature writes it.

**Status reminder**: this is **our** elementary computation. Family 236 states the *exact* result (inequality, equality included) and proves it; we check that the algebra of the quoted threshold is consistent and that it is not the magnetisation one. That is not the same thing, and the notebook does not pretend otherwise.

## 5. The lake's finite objects — seeing the theory on $C_3$ and $C_4$

Before concluding, let us come back down from the infinite graph to the finite objects the lake exposes: the cycle $C_3$ and the cycle $C_4$. They do not percolate (a finite cycle has no infinite component), but they carry the exact definitions — open boundary, closed set, connectivity — and let us **see** the statements on cases we can enumerate.

In [5]:
#check Percolation.card_edge_C3
#check Percolation.C3_closed_iff
#check Percolation.boundary_card_C4_opposite
#check Percolation.boundary_attains_min_C4

#check Percolation.card_edge_C3
──────▶  Percolation.card_edge_C3 : Fintype.card (Edge C3) = 3
#check Percolation.C3_closed_iff
──────▶  Percolation.C3_closed_iff (A : Set (Fin 3)) : openEdgeClosed C3 full3 A ↔ A = Set.univ ∨ A = ∅
#check Percolation.boundary_card_C4_opposite
──────▶  Percolation.boundary_card_C4_opposite : (boundary C4 full4 {0, 2}).card = 4
#check Percolation.boundary_attains_min_C4
──────▶  Percolation.boundary_attains_min_C4 : ∃ A, A.Nonempty ∧ A ≠ Finset.univ ∧ (boundary C4 full4 A).card = 2
--% env 4

Raw input:
{"cmd": "#check Percolation.card_edge_C3\n#check Percolation.C3_closed_iff\n#check Percolation.boundary_card_C4_opposite\n#check Percolation.boundary_attains_min_C4", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "Percolation.card_edge_C3 : Fintype.card (Edge C3) = 3"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Percolation.C3_closed_iff (A : Set (Fin 3)) : openEdgeClosed C3 full3 A ↔ A = Set.univ ∨ A = ∅"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Percolation.boundary_card_C4_opposite : (boundary C4 full4 {0, 2}).card = 4"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "Percolation.boundary_attains_min_C4 : ∃ A, A.Nonempty ∧ A ≠ Finset.univ ∧ (boundary C4 full4 A).card = 2"}],
 "env": 4}

### Reading the result

`card_edge_C3` counts the edges of the triangle: $3$, as expected. `C3_closed_iff` is the most telling result of the batch — it characterises **exactly** the closed sets of the triangle, that is, those no open edge crosses. On a three-vertex graph this characterisation can be stated and fully verified: it is the smallest laboratory in which one can look at a "boundary" without cheating.

`boundary_card_C4_opposite` is the notebook's pedagogical counter-example: in $C_4$, the boundary of two **opposite** vertices has $4$ edges, whereas that of two **adjacent** vertices has only $2$. The boundary is therefore not a function of the set's size — geometry matters. It is this kind of asymmetry that, on the infinite tree, becomes the difference between percolating and not percolating.

`boundary_attains_min_C4` closes the arc: on $C_4$, the minimum boundary is **attained**, and it is attained by a specific set. An existence statement, verified by the lake.

## Exercises

### Exercise 1 — Recover $p_c$ without the computation (on paper)

We obtained $p_c = 1/(d-1)$ from the mean-offspring criterion. Recover it **directly** on the recursion $q_{n+1} = ((1-p) + p\,q_n)^{d-1}$: show that the tangent at $q = 0$ has slope $(d-1)p$, and that the fixed point $q = 1$ ceases to be attracting exactly when that slope exceeds $1$. Check on $b = 2$: at $p = 0.5$ the slope is $1$ (critical case), at $p = 0.6$ it is $1.2$ (super-critical). Then confront your prediction with the `#eval`s of section 3 — is the agreement exact?

In [6]:
-- Exercice 1 -- retrouver p_c = 1/(d-1) par la pente en q = 0
--
-- Etape 1 : la pente en q = 0 de q |-> ((1-p) + p*q)^b vaut b*p ; justifier.
-- Etape 2 : q = 1 cesse d'etre attractif des que b*p > 1 ; en deduire
--            p_c = 1/b, puis p_c = 1/(d-1) avec b = d - 1.
-- Etape 3 : confronter la prediction aux `#eval` de la section 3 (b = 2 :
--            p = 0.5 rend 0.954145, p = 0.6 rend 0.444444). L'accord ?
--
-- TODO etudiant : remplacer le corps de `penteEnZero`, puis re-executer.

def penteEnZero (p : Float) (b : Nat) : Float :=
  Float.ofNat b * p * 0.0  -- TODO etudiant : retirer le facteur `* 0.0`

#eval penteEnZero 0.30 2    -- une fois rempli : 0.600000 (sous-critique)
#eval penteEnZero 0.50 2    -- une fois rempli : 1.000000 (cas critique)
#eval penteEnZero 0.60 2    -- une fois rempli : 1.200000 (sur-critique)

-- Exercice 1 -- retrouver p_c = 1/(d-1) par la pente en q = 0
--
-- Etape 1 : la pente en q = 0 de q |-> ((1-p) + p*q)^b vaut b*p ; justifier.
-- Etape 2 : q = 1 cesse d'etre attractif des que b*p > 1 ; en deduire
--            p_c = 1/b, puis p_c = 1/(d-1) avec b = d - 1.
-- Etape 3 : confronter la prediction aux `#eval` de la section 3 (b = 2 :
--            p = 0.5 rend 0.954145, p = 0.6 rend 0.444444). L'accord ?
--
-- TODO etudiant : remplacer le corps de `penteEnZero`, puis re-executer.

def penteEnZero (p : Float) (b : Nat) : Float :=
  Float.ofNat b * p * 0.0  -- TODO etudiant : retirer le facteur `* 0.0`

#eval penteEnZero 0.30 2    -- une fois rempli : 0.600000 (sous-critique)
─────▶  0.000000
#eval penteEnZero 0.50 2    -- une fois rempli : 1.000000 (cas critique)
─────▶  0.000000
#eval penteEnZero 0.60 2    -- une fois rempli : 1.200000 (sur-critique)
─────▶  0.000000
--% env 5

Raw input:
{"cmd": "-- Exercice 1 -- retrouver p_c = 1/(d-1) par la pente en q = 0\n--\n-- Etape 1 : la pente en q = 0 de q |-> ((1-p) + p*q)^b vaut b*p ; justifier.\n-- Etape 2 : q = 1 cesse d'etre attractif des que b*p > 1 ; en deduire\n--            p_c = 1/b, puis p_c = 1/(d-1) avec b = d - 1.\n-- Etape 3 : confronter la prediction aux `#eval` de la section 3 (b = 2 :\n--            p = 0.5 rend 0.954145, p = 0.6 rend 0.444444). L'accord ?\n--\n-- TODO etudiant : remplacer le corps de `penteEnZero`, puis re-executer.\n\ndef penteEnZero (p : Float) (b : Nat) : Float :=\n  Float.ofNat b * p * 0.0  -- TODO etudiant : retirer le facteur `* 0.0`\n\n#eval penteEnZero 0.30 2    -- une fois rempli : 0.600000 (sous-critique)\n#eval penteEnZero 0.50 2    -- une fois rempli : 1.000000 (cas critique)\n#eval penteEnZero 0.60 2    -- une fois rempli : 1.200000 (sur-critique)", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 5},
   "data": "0.000000"}],
 "env": 5}

### Exercise 2 — The super-critical fixed point, by hand then in Lean

For $b = 2$ and $p = 0.6$ we announced $q^* = 4/9$. (a) Redo the solution: $q = (0.4 + 0.6q)^2$, expand, show the equation reduces to $9q^2 - 13q + 4 = 0$. (b) Factorise and identify the two roots. (c) Why is the relevant root $4/9$ and not $1$? (d) Add a `#eval` iterating $200$ generations instead of $80$: does the value change? What does that tell you about convergence to a **stable** fixed point?

In [7]:
-- Exercice 2 -- le point fixe sur-critique : 4/9, a la main puis en Lean
--
-- Etape 1 : resoudre q = (0.4 + 0.6*q)^2 ; montrer 9q^2 - 13q + 4 = 0.
-- Etape 2 : factoriser et identifier les deux racines (1 et 4/9).
-- Etape 3 : justifier que la racine pertinente est 4/9, et non 1.
-- Etape 4 : iterer 200 generations au lieu de 80, puis comparer.
--
-- TODO etudiant : remplacer la valeur de `qSurCritique` par le point fixe
-- non trivial, puis confronter a la valeur a 80 generations (`0.444444`).

def qSurCritique : Float :=
  0.0  -- TODO etudiant : q tel que q = (0.4 + 0.6*q)^2, et q < 1

#eval qSurCritique
#check @extinction
-- TODO etudiant : ajouter `#eval extinction 0.60 2 200`

-- Exercice 2 -- le point fixe sur-critique : 4/9, a la main puis en Lean
--
-- Etape 1 : resoudre q = (0.4 + 0.6*q)^2 ; montrer 9q^2 - 13q + 4 = 0.
-- Etape 2 : factoriser et identifier les deux racines (1 et 4/9).
-- Etape 3 : justifier que la racine pertinente est 4/9, et non 1.
-- Etape 4 : iterer 200 generations au lieu de 80, puis comparer.
--
-- TODO etudiant : remplacer la valeur de `qSurCritique` par le point fixe
-- non trivial, puis confronter a la valeur a 80 generations (`0.444444`).

def qSurCritique : Float :=
  0.0  -- TODO etudiant : q tel que q = (0.4 + 0.6*q)^2, et q < 1

#eval qSurCritique
─────▶  0.000000
#check @extinction
──────▶  extinction : Float → ℕ → ℕ → Float
-- TODO etudiant : ajouter `#eval extinction 0.60 2 200`
--% env 6

Raw input:
{"cmd": "-- Exercice 2 -- le point fixe sur-critique : 4/9, a la main puis en Lean\n--\n-- Etape 1 : resoudre q = (0.4 + 0.6*q)^2 ; montrer 9q^2 - 13q + 4 = 0.\n-- Etape 2 : factoriser et identifier les deux racines (1 et 4/9).\n-- Etape 3 : justifier que la racine pertinente est 4/9, et non 1.\n-- Etape 4 : iterer 200 generations au lieu de 80, puis comparer.\n--\n-- TODO etudiant : remplacer la valeur de `qSurCritique` par le point fixe\n-- non trivial, puis confronter a la valeur a 80 generations (`0.444444`).\n\ndef qSurCritique : Float :=\n  0.0  -- TODO etudiant : q tel que q = (0.4 + 0.6*q)^2, et q < 1\n\n#eval qSurCritique\n#check @extinction\n-- TODO etudiant : ajouter `#eval extinction 0.60 2 200`", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 6},
   "data": "extinction : Float → ℕ → ℕ → Float"}],
 "env": 6}

### Exercise 3 — Separating the two thresholds (on paper)

Section 4 claims that the reconstruction threshold $(d-1)^{-1/2}$ is **strictly greater** than the magnetisation threshold $1/(d-1)$ for every $d \ge 3$. Prove it on paper: square, reduce to a common denominator, and conclude by a monotonicity argument. What is the ratio of the two thresholds as $d \to \infty$? Interpret: what becomes of the "zero magnetisation, reconstruction possible" interval on a tree of very large degree?

In [8]:
-- Exercice 3 -- separer les deux seuils (d-1)^(-1/2) et 1/(d-1)
--
-- Etape 1 : montrer (d-1)^(-1/2) > 1/(d-1) pour tout d >= 3 -- elever au carre.
-- Etape 2 : rapport des deux seuils quand d tend vers l'infini.
-- Etape 3 : interpreter l'intervalle « aimantation nulle, reconstruction
--            possible » sur un arbre de tres grand degre.
--
-- TODO etudiant : remplacer le corps de `rapportSeuils`, puis re-executer.

def rapportSeuils (d : Nat) : Float :=
  seuilReconstruction d / seuilAimantation d * 0.0  -- TODO : retirer `* 0.0`

#eval rapportSeuils 3      -- une fois rempli : 1.414214 (racine de 2)
#eval rapportSeuils 10     -- une fois rempli : 3.000000
#check @seuilReconstruction
#check @seuilAimantation

-- Exercice 3 -- separer les deux seuils (d-1)^(-1/2) et 1/(d-1)
--
-- Etape 1 : montrer (d-1)^(-1/2) > 1/(d-1) pour tout d >= 3 -- elever au carre.
-- Etape 2 : rapport des deux seuils quand d tend vers l'infini.
-- Etape 3 : interpreter l'intervalle « aimantation nulle, reconstruction
--            possible » sur un arbre de tres grand degre.
--
-- TODO etudiant : remplacer le corps de `rapportSeuils`, puis re-executer.

def rapportSeuils (d : Nat) : Float :=
  seuilReconstruction d / seuilAimantation d * 0.0  -- TODO : retirer `* 0.0`

#eval rapportSeuils 3      -- une fois rempli : 1.414214 (racine de 2)
─────▶  0.000000
#eval rapportSeuils 10     -- une fois rempli : 3.000000
─────▶  0.000000
#check @seuilReconstruction
──────▶  seuilReconstruction : ℕ → Float
#check @seuilAimantation
──────▶  seuilAimantation : ℕ → Float
--% env 7

Raw input:
{"cmd": "-- Exercice 3 -- separer les deux seuils (d-1)^(-1/2) et 1/(d-1)\n--\n-- Etape 1 : montrer (d-1)^(-1/2) > 1/(d-1) pour tout d >= 3 -- elever au carre.\n-- Etape 2 : rapport des deux seuils quand d tend vers l'infini.\n-- Etape 3 : interpreter l'intervalle \u00ab aimantation nulle, reconstruction\n--            possible \u00bb sur un arbre de tres grand degre.\n--\n-- TODO etudiant : remplacer le corps de `rapportSeuils`, puis re-executer.\n\ndef rapportSeuils (d : Nat) : Float :=\n  seuilReconstruction d / seuilAimantation d * 0.0  -- TODO : retirer `* 0.0`\n\n#eval rapportSeuils 3      -- une fois rempli : 1.414214 (racine de 2)\n#eval rapportSeuils 10     -- une fois rempli : 3.000000\n#check @seuilReconstruction\n#check @seuilAimantation", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 6},
   "data": "seuilReconstruction : ℕ → Float"},
  {"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 6},
   "data": "seuilAimantation : ℕ → Float"}],
 "env": 7}

## Conclusion

This notebook held the three registers of the Origami digestion:

1. **Our own foundation** — the Harris–Kleitman inequalities, proved in our `percolation_lean` lake, with `#check` and `#print axioms` in support: status **ours**, footprint `[propext, Classical.choice, Quot.sound]`.
2. **The quoted 2026 families** — 213 (Benjamini–Schramm criticality) and 236 (factor-of-IID threshold of Ising on the tree): statements faithfully reproduced, status **formalised** (by the authors), `lean/docs/NNN.md` skeletons as references — never executed or re-proved here.
3. **Our contribution** — two elementary yet **exact** computations, executed by the compiler: the critical probability $p_c = 1/(d-1)$ of percolation on the $d$-regular tree, with its non-trivial fixed point checkable by hand ($4/9$ for $d=3$, $p=0.6$); and the clarification of the **two distinct thresholds** of Ising on the tree, with their Kesten–Stigum identity verified numerically.

**SOTA verdicts** (repository rule): **SOTA-OK** for this notebook — real Lean 4 + Mathlib, real compiled lake, real native `#eval`s, no substitution. **INTRINSIC** for re-proving the 2026 families: beyond this repository's scale — the existence of the published skeletons is precisely what makes quoting the honest move.

**References**: lake [`percolation_lean`](../../../Probas/Applications/Percolation/percolation_lean/) · issue #19898 (Origami Epic) and #19920 (Fold 3) · Harris, *A lower bound for the critical probability in a certain percolation process* (1960) · Fortuin, Kasteleyn & Ginibre (1971) · Kesten & Stigum (1966) · Lyons (the factor-of-IID threshold question) · families 213 and 236 of the openai/math repository (2026) and their `lean/docs/NNN.md` skeletons.